## Configuration <a name="Config"></a>
Building a connection to API wykop.pl

In [ ]:
import pandas as pd
from wykop_sdk_reloaded.v3.client import AuthClient, WykopApiClient
from wykop_sdk_reloaded.v3.types import StreamSortType, RequestType
from wykop_sdk_reloaded.exceptions import WykopApiNotFoundError

API_key = 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'
secret_key = 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'

url = "https://wykop.pl/api/v3"

auth = AuthClient()
auth.authenticate_app(API_key, secret_key)

api = WykopApiClient(auth)

## Tag 'polska'
    

In [2]:
tags = {}
tags_details = api.tags_get_popular_tags()
for tag in tags_details['data']:
    tags[tag['name']] = tag['followers']

print(dict(sorted(tags.items(), key=lambda item: item[1])))


{'pdk': 627, 'pytanie': 845, 'usa': 1227, '4konserwy': 1745, 'neuropa': 2023, 'rozowepaski': 3075, 'f1': 8895, 'gielda': 24994, 'mecz': 29383, 'nieruchomosci': 49016, 'humorobrazkowy': 54230, 'muzyka': 71585, 'heheszki': 105012, 'polska': 128562, 'ciekawostki': 140762}


## Collecting links and their meta data from January to March 2026. 

In [3]:
tag_polska_list = []
links = set()

for month in [1,2,3]:
    page = 1
    while True:
        tag_polska = api.tags_get_stream_of_tag('polska',StreamSortType.BEST,page=str(page),limit=None,year=2026,month=month)

        if not tag_polska['data']:
            break

        for link in tag_polska['data']:
            if len(str(link['id'])) == 7: # links have 7 digit ID
                links.add(link['id'])
                tag_polska_list.append(link)

        page += 1

print(len(links))

1088


### Searching for users that downvote links.

Downloading metadata of all links.

In [4]:
result = {}
for l in links:
    downvotes_metadata = api.raw_request(url+f"/links/{l}/upvotes/down?page=1", RequestType.GET)

    result[l] = {
        "voted_user": [],
        "voted_time": [],
        "voted_reason": [],
    }

    for i in downvotes_metadata['data']:
        result[l]["voted_time"].append(i["created_at"])
        result[l]["voted_reason"].append(i["reason"])
        result[l]["voted_user"].append(i["user"]["username"])

Saving links metadata to the table.

In [5]:
rows = []
for link_id, data in result.items():
    rows.append(
        {
            "id": link_id,
            "voted_user": data["voted_user"],     # list
            "voted_time": data["voted_time"],     # list
            "voted_reason": data["voted_reason"], # list
        }
    )

df_downvotes = pd.DataFrame(rows)

In [6]:
display(df_downvotes)

,id,voted_user,voted_time,voted_reason
0,7901187,"[rychu-nalepa, Njal, HardWax, CzteryTrzeciePiR...","[2026-03-04 23:02:40, 2026-03-04 22:58:26, 202...","[duplicate, duplicate, duplicate, duplicate, s..."
1,7911431,"[wpiszlogintwardzielu, grubson234567, IbraKa, ...","[2026-03-22 11:40:53, 2026-03-23 15:26:45, 202...","[duplicate, spam, spam, spam, spam, spam, spam..."
2,7905287,"[prawilny_prawak, muckfods, Ovis_aries, wkplog...","[2026-03-11 17:35:07, 2026-03-11 16:27:30, 202...","[spam, spam, fake, fake, fake, fake, fake, fak..."
3,7880715,"[Vort, Lipathor, Zajec5, LoginZajetyPrzezKomor...","[2026-01-31 12:17:39, 2026-01-31 15:01:49, 202...","[spam, fake, invalid, invalid]"
4,7884811,"[ponzbi, sebpsx, rychu-nalepa, caveflacon, jar...","[2026-02-06 23:06:25, 2026-02-07 08:14:34, 202...","[duplicate, spam, spam, fake, fake, invalid, i..."
...,...,...,...,...
1083,7895027,"[zagubionychromosom, szkutnik, Spust, Arystokr...","[2026-02-23 11:25:01, 2026-02-24 07:50:44, 202...","[duplicate, spam, spam, spam, spam, spam, spam..."
1084,7905269,"[er6f, wyrwiflak, Kartoffelpanzerkampfwagen, G...","[2026-03-12 09:12:37, 2026-03-11 20:55:47, 202...","[spam, spam, spam, spam, spam, spam, spam, spa..."
1085,7909367,"[Bertrandy, asterrr, Reiter1906, priom, prawil...","[2026-03-19 01:05:40, 2026-03-19 21:35:19, 202...","[duplicate, spam, spam, spam, spam, spam, spam..."
1086,7884795,"[Epikur, Nie_umre_za_ciebie, Frater_Lucifer, A...","[2026-02-07 13:09:02, 2026-02-08 20:01:32, 202...","[spam, fake, fake, fake, fake, fake, fake, fak..."


Saving copy to the table.

In [7]:
df_downvotes.to_json('downvotes_01-03.json', index=True,)

### Searching for users that upvote links.

In [8]:
result = {}
for l in links:
    downvotes_metadata = api.raw_request(url+f"/links/{l}/upvotes/up?page=1", RequestType.GET)

    result[l] = {
        "voted_user": [],
        "voted_time": []
    }

    for i in downvotes_metadata['data']:
        result[l]["voted_time"].append(i["created_at"])
        result[l]["voted_user"].append(i["user"]["username"])

In [9]:
rows = []

for link_id, data in result.items():
    rows.append(
        {
            "id": link_id,
            "up_voted_user": data["voted_user"],
            "up_voted_time": data["voted_time"],
        }
    )

df_upvotes = pd.DataFrame(rows)

In [10]:
display(df_upvotes)

,id,up_voted_user,up_voted_time
0,7901187,"[mac4479, Poldi7777, szoorstki, dfpbld, andry0...","[2026-03-30 22:06:15, 2026-03-16 15:45:30, 202..."
1,7911431,"[ferrrnando, kirii-tips, pascal256, piromanu, ...","[2026-03-28 11:20:17, 2026-03-27 22:39:49, 202..."
2,7905287,"[Sanya, vipr78, kirson, Kismeth, gr4chojnice, ...","[2026-03-17 06:50:29, 2026-03-16 22:10:06, 202..."
3,7880715,"[h3xxx, kuboko, faster2006, vipr78, Sidroff, c...","[2026-02-08 23:43:54, 2026-02-04 21:09:20, 202..."
4,7884811,"[piromanu, robo_epa, grzegkr, VishnyaVishnevsk...","[2026-02-28 23:40:50, 2026-02-16 15:34:53, 202..."
...,...,...,...
1083,7895027,"[kopiret, Adam650, paul43, rominciarz, Dumle00...","[2026-03-13 23:23:10, 2026-03-08 13:00:38, 202..."
1084,7905269,"[filios, bober-men, pcxelja, rukh, Booonzo, mr...","[2026-03-30 18:51:25, 2026-03-30 12:22:32, 202..."
1085,7909367,"[kirson, higi17, marcin1100, kowcio11, enedue,...","[2026-03-25 10:46:35, 2026-03-24 21:06:58, 202..."
1086,7884795,"[robo_epa, nadmuchane_jaja, Szyntyn, VishnyaVi...","[2026-02-16 15:36:15, 2026-02-13 02:46:36, 202..."


In [11]:
df_upvotes.to_csv('upvotes_01-03.csv', index=True,)

### Creating a table with downvoting users metadata.

In [12]:
link_id = []
created_at = []
title = []
author = []
published_at = []
votes_up = []
votes_down = []
votes_total = []
comment_count = []
tags = []

for i in tag_polska_list:
    link_id.append(i['id'])
    created_at.append(i['created_at'])
    title.append(i['title'])
    author.append(i['author']['username'])
    published_at.append(i['published_at'])
    votes_up.append(i['votes']['up'])
    votes_down.append(i['votes']['down'])
    votes_total.append(i['votes']['count'])
    tags.append(i['tags'])

len(created_at)

1088

In [13]:
df = pd.DataFrame({
    'id': link_id,
    'created_at': created_at,
    'title': title,
    'author': author,
    'published_at': published_at,
    'votes_up': votes_up,
    'votes_down': votes_down,
    'votes_total': votes_total,
    'tags': tags,
})

In [14]:
display(df)

,id,created_at,title,author,published_at,votes_up,votes_down,votes_total,tags
0,7881315,2026-01-31 20:16:49,Kanał Autostrady Polska krytykuje planowany pr...,JKL789,2026-02-01 12:21:30,360,4,356,"[wroclaw, autostradypolska, autostrady, polski..."
1,7881129,2026-01-31 14:40:47,Pies zaatakował 11-latkę wracającą ze szkoły. ...,VoxClamantisInDeserto,2026-01-31 21:53:44,591,11,580,"[psy, psiarze, polska, wydarzenia, spoleczenstwo]"
2,7881085,2026-01-31 13:18:42,Kraków. Lotnisko zrzuca toksyczne odpady do po...,paramedix,2026-01-31 16:18:51,453,6,447,"[polska, krakow, lotnisko, lotnictwo, ekologia..."
3,7881055,2026-01-31 12:37:52,Od 2028 legalne retrofity w starszych samochod...,B.....r,2026-02-01 10:08:02,283,4,279,"[samochody, prawo, polska, dowod, drogi]"
4,7881051,2026-01-31 12:34:06,To nie żart. W centrum Poznania stanęły ogrzew...,InfoMagazyn,2026-01-31 22:35:24,282,60,222,"[poznan, polska, wydarzenia, mrozy, psp]"
...,...,...,...,...,...,...,...,...,...
1083,7899025,2026-03-01 16:52:41,Choroby cywilizacyjne i rak czy koncerny spoży...,robert-i,2026-03-02 08:06:55,391,83,308,"[zdowie, choroba, medycyna, jedzenie, ciekawos..."
1084,7898981,2026-03-01 14:31:57,"Dla Bydgoszczy likwidacja, dla Torunia nowa uc...",bylemtampotwierdzam,2026-03-01 21:07:36,549,14,535,"[calbecki, bydgoszcz, kujawskopomorskie, torun..."
1085,7898949,2026-03-01 13:05:48,15 Lat temu w Lesie Kabackim brutalnie zamordo...,NotourMATT,2026-03-01 15:13:07,1482,0,1482,"[polska, afera, aferareprywatyzacyjna, brzeska..."
1086,7898853,2026-03-01 09:57:34,Prywatna wizyta u lekarza? Zapłacisz nawet 800 zł,iggy_p,2026-03-01 18:51:57,627,8,619,[polska]


### Full left join of links table and downvoting/upvoting users metadata table.

In [15]:
import functools as ft
dfs = [df, df_downvotes, df_upvotes]
df_final = ft.reduce(lambda left,right: pd.merge(left, right, on='id'), dfs)


In [10]:
display(df_final)

,Unnamed: 0,id,created_at,title,author,published_at,votes_up,votes_down,votes_total,tags,voted_user,voted_time,voted_reason,up_voted_user,up_voted_time
0,0,7881315,2026-01-31 20:16:49,Kanał Autostrady Polska krytykuje planowany pr...,JKL789,2026-02-01 12:21:30,360,4,356,"['wroclaw', 'autostradypolska', 'autostrady', ...","['P.....a', 'Nudel1306', 'Njal', 'johny-bravooo']","['2026-02-01 16:02:53', '2026-01-31 21:42:06',...","['spam', 'wrong', 'invalid', 'invalid']","['sbb', 'Fz7dA4pV', 'Bagol', 'wykopkiewicz', '...","['2026-03-02 14:53:45', '2026-02-06 23:23:17',..."
1,1,7881129,2026-01-31 14:40:47,Pies zaatakował 11-latkę wracającą ze szkoły. ...,VoxClamantisInDeserto,2026-01-31 21:53:44,591,11,580,"['psy', 'psiarze', 'polska', 'wydarzenia', 'sp...","['xx3r4xx', 'szatanizmo', 'Fikster', 'h86Ilk',...","['2026-02-01 12:10:21', '2026-02-01 11:09:59',...","['spam', 'spam', 'spam', 'spam', 'spam', 'spam...","['Varcetti', 'marcin1100', 'Bochnovic', 'maror...","['2026-02-14 17:53:07', '2026-02-05 22:07:22',..."
2,2,7881085,2026-01-31 13:18:42,Kraków. Lotnisko zrzuca toksyczne odpady do po...,paramedix,2026-01-31 16:18:51,453,6,447,"['polska', 'krakow', 'lotnisko', 'lotnictwo', ...","['prawilny_prawak', 'CzteryTrzeciePiRdoTrzecie...","['2026-01-31 17:20:36', '2026-01-31 14:48:11',...","['spam', 'spam', 'fake', 'fake', 'invalid', 'i...","['kuboko', 'wolownik', 'BlueScreeen', 'Mister1...","['2026-02-06 23:15:21', '2026-02-06 19:58:21',..."
3,3,7881055,2026-01-31 12:37:52,Od 2028 legalne retrofity w starszych samochod...,B.....r,2026-02-01 10:08:02,283,4,279,"['samochody', 'prawo', 'polska', 'dowod', 'dro...","['fremmm', 'Beheris', 'Ptysiu_323', 'Njal']","['2026-02-01 10:40:37', '2026-01-31 13:13:22',...","['spam', 'fake', 'invalid', 'invalid']","['fuxxiasty', 'lkhjgkuyr', 'RRb83', 'lukasluk1...","['2026-02-04 00:29:23', '2026-02-03 22:36:49',..."
4,4,7881051,2026-01-31 12:34:06,To nie żart. W centrum Poznania stanęły ogrzew...,InfoMagazyn,2026-01-31 22:35:24,282,60,222,"['poznan', 'polska', 'wydarzenia', 'mrozy', 'p...","['pablo4096', 'estampida', 'xx3r4xx', 'zubrzpu...","['2026-02-02 10:38:07', '2026-02-01 14:46:00',...","['spam', 'spam', 'spam', 'spam', 'spam', 'spam...","['Devilus', 'fuxxiasty', 'BlueScreeen', 'Paxx8...","['2026-02-04 04:24:06', '2026-02-04 00:31:12',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1083,1083,7899025,2026-03-01 16:52:41,Choroby cywilizacyjne i rak czy koncerny spoży...,robert-i,2026-03-02 08:06:55,391,83,308,"['zdowie', 'choroba', 'medycyna', 'jedzenie', ...","['iks-ski', 'Nikczemny_Pomidor', 'KosmicznyNal...","['2026-03-05 22:02:06', '2026-03-03 09:12:03',...","['spam', 'spam', 'spam', 'spam', 'spam', 'spam...","['Dzordzi', 'gasu_kurobikari', 'Rilect', 'Hash...","['2026-03-09 10:54:38', '2026-03-05 20:43:21',..."
1084,1084,7898981,2026-03-01 14:31:57,"Dla Bydgoszczy likwidacja, dla Torunia nowa uc...",bylemtampotwierdzam,2026-03-01 21:07:36,549,14,535,"['calbecki', 'bydgoszcz', 'kujawskopomorskie',...","['lul3k', 'Lysy32', 'ufo666', 'xxvv', 'jarl_br...","['2026-03-02 20:28:12', '2026-03-02 18:18:23',...","['fake', 'fake', 'fake', 'fake', 'fake', 'fake...","['p00tinMalpaRU', 'eSUBA94', 'Ashkhan', 'Dzord...","['2026-04-03 05:38:06', '2026-04-02 16:40:03',..."
1085,1085,7898949,2026-03-01 13:05:48,15 Lat temu w Lesie Kabackim brutalnie zamordo...,NotourMATT,2026-03-01 15:13:07,1482,0,1482,"['polska', 'afera', 'aferareprywatyzacyjna', '...",[],[],[],"['Dzordzi', 'lysy_orangutan', 'kondzio2003', '...","['2026-03-09 10:56:21', '2026-03-08 14:31:00',..."
1086,1086,7898853,2026-03-01 09:57:34,Prywatna wizyta u lekarza? Zapłacisz nawet 800 zł,iggy_p,2026-03-01 18:51:57,627,8,619,['polska'],"['mezon', 'efilist', 'mppmpmp', 'FuPa', 'Recke...","['2026-03-05 12:36:25', '2026-03-02 12:08:33',...","['spam', 'spam', 'spam', 'spam', 'fake', 'fake...","['Dzordzi', 'ferrrnando', 'kyniu6', 'wilddd', ...","['2026-03-09 10:56:05', '2026-03-08 08:15:17',..."


In [17]:
df_final.to_csv('df_final_01-03.csv', index=True)

## Creating downvoters table.

In [17]:
import pandas as pd
import ast

df = pd.read_csv("df_final_01-03.csv")

def to_list(s):
    if pd.isna(s):
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

# 1) Parse list-like string columns
for col in ["voted_user", "voted_time", "voted_reason"]:
    df[col] = df[col].apply(to_list)

# 2) Build a single list of triples per row: [(user, time, reason), ...]
def zip_votes(row):
    users = row["voted_user"]
    times = row["voted_time"]
    reasons = row["voted_reason"]
    # zip will only go up to the shortest list length -> safe alignment
    return list(zip(users, times, reasons))

df["votes_triplets"] = df.apply(zip_votes, axis=1)

# 3) Explode once on the list of triples
neg = df[["id", "votes_triplets"]].explode("votes_triplets", ignore_index=True)

# 4) Split the triple back into separate columns
neg[["user", "time", "reason"]] = pd.DataFrame(
    neg["votes_triplets"].tolist(),
    index=neg.index
)

neg = neg.drop(columns=["votes_triplets"])

neg["time"] = pd.to_datetime(neg["time"], errors="coerce")
neg["date"] = neg["time"].dt.date
neg["hour"] = neg["time"].dt.hour

neg["votes_night"] = ((neg["hour"] >= 0) & (neg["hour"] < 6)).astype(int)
neg["votes_day"] = ((neg["hour"] >= 6) & (neg["hour"] <= 23)).astype(int)

neg["time"] = neg["time"].dt.time
neg = neg.drop(columns=["hour"])
# 5) Optional cleanup
neg = neg[neg["user"].notna()]          # drop empty
neg = neg.drop_duplicates(["id", "user"])  # one row per (id, user) if you want

# 6) Save
neg.to_csv("negative_votes_long_fixed.csv", index=False)

In [18]:
display(neg)

,id,user,time,reason,date,votes_night,votes_day
0,7881315,P.....a,16:02:53,spam,2026-02-01,0,1
1,7881315,Nudel1306,21:42:06,wrong,2026-01-31,0,1
2,7881315,Njal,06:44:02,invalid,2026-02-01,0,1
3,7881315,johny-bravooo,20:21:24,invalid,2026-01-31,0,1
4,7881129,xx3r4xx,12:10:21,spam,2026-02-01,0,1
...,...,...,...,...,...,...,...
21317,7898769,xN0M4D,17:24:45,invalid,2026-03-01,0,1
21318,7898769,grzesiu,17:14:16,invalid,2026-03-01,0,1
21319,7898769,PaganMuffin,17:11:43,invalid,2026-03-01,0,1
21320,7898769,BuszujacyWZbozu112,16:54:26,invalid,2026-03-01,0,1


## Creating upvoters table.

In [19]:
# Parse list-like string columns for upvotes
for col in ["up_voted_user", "up_voted_time"]:
    df[col] = df[col].apply(to_list)

# Build a single list of pairs per row: [(user, time), ...]
def zip_upvotes(row):
    users = row["up_voted_user"]
    times = row["up_voted_time"]
    # zip keeps elements aligned by position and stops at shortest list
    return list(zip(users, times))

df["up_votes_pairs"] = df.apply(zip_upvotes, axis=1)

# Explode once on the list of (user, time) pairs
up = df[["id", "up_votes_pairs"]].explode("up_votes_pairs", ignore_index=True)

# Split the pair back into separate columns
up[["user", "time"]] = pd.DataFrame(
    up["up_votes_pairs"].tolist(),
    index=up.index
)

up = up.drop(columns=["up_votes_pairs"])

up["time"] = pd.to_datetime(up["time"], errors="coerce")
up["date"] = up["time"].dt.date
up["time"] = up["time"].dt.time

# Optional cleanup
up = up[up["user"].notna()]           # drop empty rows
up = up.drop_duplicates(["id", "user"])  # one row per (id, user) if desired

# Save or use in memory
up.to_csv("up_votes_long_fixed.csv", index=False)

In [20]:
display(up)

,id,user,time,date,votes_night,votes_day
0,7881315,sbb,14:53:45,2026-03-02,0,1
1,7881315,Fz7dA4pV,23:23:17,2026-02-06,0,1
2,7881315,Bagol,22:05:55,2026-02-05,0,1
3,7881315,wykopkiewicz,17:33:10,2026-02-04,0,1
4,7881315,piromanu,23:30:06,2026-02-03,0,1
...,...,...,...,...,...,...
761048,7898769,patryk-wuwuw,07:12:32,2026-03-01,0,1
761049,7898769,dziacha,06:25:42,2026-03-01,0,1
761050,7898769,11Sauger18v,06:04:51,2026-03-01,0,1
761051,7898769,sj410,05:49:36,2026-03-01,1,0


## Joining downvoters and upvoters tables.

In [31]:
# 1) Downvotes per user
downvotes_per_user = (
    neg.groupby("user", as_index=False)
       .agg(
           downvotes=("user", "size"),
           downvotes_day=("votes_day", "sum"),
           downvotes_night=("votes_night", "sum"),
       )
)

# 2) Upvotes per user
upvotes_per_user = (
    up.groupby("user", as_index=False)
      .agg(
          upvotes=("user", "size"),
          upvotes_day=("votes_day", "sum"),
          upvotes_night=("votes_night", "sum"),
      )
)

# 3) Combine into one DataFrame
user_votes = pd.merge(
    downvotes_per_user,
    upvotes_per_user,
    on="user",
    how="outer"
).fillna(0)

int_cols = [
    "downvotes", "downvotes_day", "downvotes_night",
    "upvotes", "upvotes_day", "upvotes_night",
]

for c in int_cols:
    user_votes[c] = user_votes[c].astype(int)

# 4) All votes per user (up + down)
user_votes["total_votes"] = (
    user_votes["downvotes_day"]
  + user_votes["downvotes_night"]
  + user_votes["upvotes_day"]
  + user_votes["upvotes_night"]
)

# 5) Night votes vs all votes
user_votes["night_votes_share"] = (
    (user_votes["downvotes_night"] + user_votes["upvotes_night"])
    / user_votes["total_votes"].replace(0, pd.NA)
)

user_votes["night_votes_share"] = round(user_votes["night_votes_share"].fillna(0),2)

# 6) Save
user_votes.to_csv("user_votes_counts.csv", index=False)
display(user_votes)

,user,downvotes,downvotes_day,downvotes_night,upvotes,upvotes_day,upvotes_night,total_votes,night_votes_share
0,------------------,0,0,0,16,16,0,16,0.00
1,-----------------------------,4,4,0,3,3,0,7,0.00
2,---polibudziaki_pl---,0,0,0,1,0,1,1,1.00
3,--__--__--,1,0,1,8,4,4,9,0.56
4,-.....-,0,0,0,1,1,0,1,0.00
...,...,...,...,...,...,...,...,...,...
33495,zzakrzak,0,0,0,304,299,5,304,0.02
33496,zzbkk,0,0,0,16,16,0,16,0.00
33497,zzlogin,0,0,0,2,2,0,2,0.00
33498,zzyzor,0,0,0,48,48,0,48,0.00


## Creating list of users (authors, voters).

In [ ]:
# df = pd.read_csv("df_final_01-03.csv")

In [34]:
all_users_from_lists = pd.Series(
    [
        user
        for col in ['voted_user', 'up_voted_user']
        for lst in df_final[col].dropna()
        for user in lst
    ]
)

authors = df_final['author'].dropna().astype(str)
all_users = pd.concat([all_users_from_lists, authors], ignore_index=True)
unique_users = all_users.drop_duplicates().tolist()

len(unique_users)

33606

In [36]:
user_profiles = []
users_deleted = []
url = "https://wykop.pl/api/v3/profile/users/"

for i, u in enumerate(unique_users, start=1):
    try:
        user_profiles.append(api.raw_request(url + str(u), RequestType.GET))
    except WykopApiNotFoundError:
        users_deleted.append(u)

    # log every 100 users
    if i % 100 == 0:
        print(f"Przetworzono {i} / {len(unique_users)} użytkowników")

print(f"Skończone. Udało się pobrać: {len(user_profiles)} profili")
print(f"Użytkownicy usunięci / niedostępni: {len(users_deleted)}")

Przetworzono 100 / 33606 użytkowników
Przetworzono 200 / 33606 użytkowników
Przetworzono 300 / 33606 użytkowników
Przetworzono 400 / 33606 użytkowników
Przetworzono 500 / 33606 użytkowników
Przetworzono 600 / 33606 użytkowników
Przetworzono 700 / 33606 użytkowników
Przetworzono 800 / 33606 użytkowników
Przetworzono 900 / 33606 użytkowników
Przetworzono 1000 / 33606 użytkowników
Przetworzono 1100 / 33606 użytkowników
Przetworzono 1200 / 33606 użytkowników
Przetworzono 1300 / 33606 użytkowników
Przetworzono 1400 / 33606 użytkowników
Przetworzono 1500 / 33606 użytkowników
Przetworzono 1600 / 33606 użytkowników
Przetworzono 1700 / 33606 użytkowników
Przetworzono 1800 / 33606 użytkowników
Przetworzono 1900 / 33606 użytkowników
Przetworzono 2000 / 33606 użytkowników
Przetworzono 2100 / 33606 użytkowników
Przetworzono 2200 / 33606 użytkowników
Przetworzono 2300 / 33606 użytkowników
Przetworzono 2400 / 33606 użytkowników
Przetworzono 2500 / 33606 użytkowników
Przetworzono 2600 / 33606 użytkown

At the time the link data—including trigger timestamps and the users who buried the links was retrieved, the accounts listed in the "users_deleted" set were active.
The "users_deleted" lists are consulted when attempting to retrieve profiles that cannot be found.
The naming convention is similar, and the accounts in question were identified as bots by Wykop's moderation team.

In [ ]:
df_final.to_csv('df_final_01-03.csv', index=True)

In [37]:
users_deleted

['P.....a',
 'd.....d',
 'K.....a',
 's.....o',
 's.....a',
 'M.....o',
 'A.....a',
 'Z.....n',
 '3.....r',
 'F....._',
 'k.....l',
 'E.....7',
 'M.....s',
 'W.....i',
 'M.....l',
 'K.....3',
 'e.....o',
 'c.....e',
 'F.....y',
 '4.....f',
 'E.....o',
 'S.....j',
 'h.....k',
 'e.....1',
 's.....v',
 'W.....n',
 'w.....e',
 'B.....a',
 's.....5',
 'c.....r',
 'T.....4',
 'w.....n',
 'C.....1',
 'E.....4',
 'E.....l',
 'M.....m',
 'd.....1',
 'n.....e',
 'f.....z',
 'z.....k',
 'h.....e',
 'f.....s',
 'P.....3',
 'K.....2',
 'U.....d',
 'g.....a',
 't.....e',
 't.....a',
 'Z.....h',
 'n.....y',
 't.....0',
 'd.....g',
 'P.....s',
 's.....m',
 'S.....i',
 'B.....r',
 'Z.....e',
 'K.....n',
 's.....l',
 'z.....i',
 's.....e',
 'T.....e',
 'A.....y',
 'o.....I',
 'a.....k',
 'I.....l',
 'Q.....z',
 'j.....i',
 'B.....k',
 'k.....t',
 'h.....o',
 'c.....x',
 'd.....b',
 'M.....k',
 'F.....k',
 'a.....0',
 'k.....o',
 's.....1',
 'n.....r',
 'Q.....m',
 'G.....a',
 'P.....5',
 'W.....e',
 'z.

In [41]:
df_users_deleted = pd.DataFrame({'user': users_deleted})

In [ ]:
suspicious_accounts_deleted =['K.....3','m.....a','M.....l','w.....n','B.....r','s.....o','E.....7','W.....n','W.....i','4.....f','E.....l','h.....k','K.....a','e.....1','n.....e','z.....k','h.....e','d.....d','w.....e','M.....m','F....._','e.....o','M.....s','s.....5','E.....o','3.....r','c.....e','C.....1','B.....a','s.....v','E.....4','s.....1','c.....r','Z.....n','k.....l','f.....s','M.....o','f.....z','A.....a','P.....a','s.....a','S.....j','P.....3','F.....y']

Retrieving selected user attributes.

In [38]:
username = []
for profile in user_profiles:
    username.append(profile["data"]["username"])

colors = []
for profile in user_profiles:
    colors.append(profile["data"]["color"])

actions = []
for profile in user_profiles:
    actions.append(profile["data"]["summary"]["actions"])

links = []
for profile in user_profiles:
    links.append(profile["data"]["summary"]["links"])

links_added = []
for profile in user_profiles:
    links_added.append(profile["data"]["summary"]["links_details"]["added"])

links_commented = []
for profile in user_profiles:
    links_commented.append(profile["data"]["summary"]["links_details"]["commented"])

links_published = []
for profile in user_profiles:
    links_published.append(profile["data"]["summary"]["links_details"]["published"])

links_related = []
for profile in user_profiles:
    links_related.append(profile["data"]["summary"]["links_details"]["related"])

links_up = [] # list of links upvoted by user
for profile in user_profiles:
    links_up.append(profile["data"]["summary"]["links_details"]["up"])

member_since = []
for profile in user_profiles:
    member_since.append(profile["data"]['member_since'])

rank = []
for profile in user_profiles:
    rank.append(profile["data"]['rank']['position'])

company = []
for profile in user_profiles:
    company.append(profile["data"]['company'])

gender = []
for profile in user_profiles:
    gender.append(profile["data"]['gender'])

verified = []
for profile in user_profiles:
    verified.append(profile["data"]['verified'])

banned = []
for profile in user_profiles:
    banned.append(profile["data"]['banned'])

name = []
for profile in user_profiles:
    name.append(profile["data"]['name'])

Creating a table with user metadata.

In [39]:
df = pd.DataFrame({
    "user": username,
    "color": colors,
    "actions": actions,
    "links": links,
    "links_added": links_added,
    "links_commented": links_commented,
    "links_published": links_published,
    "links_related": links_related,
    "links_up": links_up,
    "member_since": member_since,
    "rank": rank,
    "gender": gender,
    "company": company,
    "verified": verified,
    "banned": banned,
    "name": name,
})
display(df)

,user,color,actions,links,links_added,links_commented,links_published,links_related,links_up,member_since,rank,gender,company,verified,banned,name
0,Nudel1306,orange,30689,5169,0,4473,0,0,696,2020-06-28 16:40:42,NaN,m,False,False,None,
1,Njal,orange,27447,9762,16,3209,5,25,6512,2023-05-12 14:26:25,NaN,m,False,False,{'reason': 'Naruszenie regulaminu - nieodpowie...,
2,johny-bravooo,orange,8709,8684,0,3974,0,2,4708,2022-08-08 14:11:01,NaN,NaN,False,False,None,Johny Bravo
3,xx3r4xx,orange,148,143,0,46,0,0,97,2025-12-18 10:36:33,NaN,m,False,False,None,
4,szatanizmo,orange,1611,693,0,650,0,0,43,2020-01-15 15:21:44,NaN,NaN,True,False,None,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33419,ObserwatorGospodarczy,burgundy,5074,5070,4706,125,745,200,39,2019-07-02 14:08:29,489.0,NaN,True,True,None,
33420,franelo,burgundy,1833,621,15,490,3,19,97,2016-01-03 00:12:17,761.0,m,False,False,None,
33421,progressbar,burgundy,1157,652,64,382,11,20,186,2014-02-05 21:58:41,556.0,m,False,False,None,
33422,AgroProfil,burgundy,697,695,543,15,27,125,12,2021-10-25 13:25:45,977.0,m,True,True,None,


In [44]:
df = pd.concat([df, df_users_deleted],ignore_index=True)

In [45]:
display(df)

,user,color,actions,links,links_added,links_commented,links_published,links_related,links_up,member_since,rank,gender,company,verified,banned,name
0,Nudel1306,orange,30689.0,5169.0,0.0,4473.0,0.0,0.0,696.0,2020-06-28 16:40:42,NaN,m,False,False,None,
1,Njal,orange,27447.0,9762.0,16.0,3209.0,5.0,25.0,6512.0,2023-05-12 14:26:25,NaN,m,False,False,{'reason': 'Naruszenie regulaminu - nieodpowie...,
2,johny-bravooo,orange,8709.0,8684.0,0.0,3974.0,0.0,2.0,4708.0,2022-08-08 14:11:01,NaN,NaN,False,False,None,Johny Bravo
3,xx3r4xx,orange,148.0,143.0,0.0,46.0,0.0,0.0,97.0,2025-12-18 10:36:33,NaN,m,False,False,None,
4,szatanizmo,orange,1611.0,693.0,0.0,650.0,0.0,0.0,43.0,2020-01-15 15:21:44,NaN,NaN,True,False,None,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33601,u.....t,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33602,t.....l,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33603,m.....p,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33604,s.....d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saving copy.

In [46]:
df.to_csv('users_final_01-03.csv', index=True)

In [65]:
df = pd.read_csv('users_final_01-03.csv', index_col=0)

In [66]:
display(user_votes)

,user,downvotes,upvotes
0,------------------,0,16
1,-----------------------------,4,3
2,---polibudziaki_pl---,0,1
3,--__--__--,1,8
4,-.....-,0,1
...,...,...,...
33495,zzakrzak,0,304
33496,zzbkk,0,16
33497,zzlogin,0,2
33498,zzyzor,0,48


In [74]:
df = pd.merge(df,user_votes, on='user',how='outer')
display(df)

,user,color,actions,links,links_added,links_commented,links_published,links_related,links_up,member_since,rank,gender,company,verified,banned,name,downvotes_1Q,upvotes_1Q
0,------------------,orange,595.0,109.0,0.0,0.0,0.0,0.0,109.0,2025-11-17 12:02:11,NaN,NaN,False,False,NaN,--- ---,0.0,16.0
1,-----------------------------,orange,4084.0,245.0,0.0,191.0,0.0,6.0,48.0,2025-09-25 10:53:00,NaN,NaN,False,False,NaN,NaN,4.0,3.0
2,---polibudziaki_pl---,orange,303.0,25.0,0.0,0.0,0.0,0.0,25.0,2024-05-16 21:37:25,NaN,m,False,False,NaN,Wojciech Radziwon,0.0,1.0
3,--__--__--,orange,62.0,27.0,0.0,0.0,0.0,0.0,27.0,2025-08-26 03:24:27,NaN,NaN,False,False,NaN,TheEldarie,1.0,8.0
4,-.....-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33601,zzakrzak,orange,32442.0,25522.0,0.0,0.0,0.0,0.0,25522.0,2015-09-22 22:04:51,NaN,m,False,False,NaN,NaN,0.0,304.0
33602,zzbkk,orange,17910.0,3366.0,42.0,1533.0,7.0,6.0,1785.0,2016-03-10 14:18:01,2507.0,m,False,False,NaN,NaN,0.0,16.0
33603,zzlogin,orange,1065.0,71.0,0.0,0.0,0.0,0.0,71.0,2023-04-29 10:35:27,NaN,m,False,False,NaN,Zygi Zygote,0.0,2.0
33604,zzyzor,orange,1425.0,909.0,0.0,14.0,0.0,0.0,895.0,2021-04-01 08:10:04,NaN,m,False,False,NaN,NaN,0.0,48.0
